# Lab 1 — Comprendre le jeu et jouer au hasard (correction)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-medecine/notebooks/pym-s8-lab-1-correction.ipynb)

*Python pour la médecine — Séance 8, projet Puissance 4.*


**Objectifs.** Comprendre les règles du Puissance 4 et la façon dont
PettingZoo représente une partie ; jouer une partie au hasard ; écrire
un premier agent qui respecte le contrat de ML-Arena ; écrire la boucle
`jouer_partie` qui fait s'affronter deux agents **exactement comme la
plateforme les appelle** ; mesurer ce que donne le hasard sur 200
parties.

**Comment travailler.** Exécutez les cellules dans l'ordre. Une cellule
marquée `# À vous` contient un squelette (`...`) à compléter ; la
cellule suivante vérifie votre réponse et affiche « ✅ Exercice N
réussi ». Les cellules « Lire du code » demandent d'écrire votre
prédiction **avant** d'exécuter.

## 0. Installation

In [ ]:
import subprocess
import sys

try:
    import pettingzoo
except ImportError:  # Colab : PettingZoo n'est pas préinstallé
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pettingzoo[classic]"], check=True)

import random
import time

import matplotlib.pyplot as plt
import numpy as np
from pettingzoo.classic import connect_four_v3

random.seed(0)
print("PettingZoo prêt")

---

## 1. Les règles

Deux joueurs lâchent à tour de rôle un pion dans l'une des 7 colonnes
d'une grille verticale de 6 lignes ; le pion tombe dans la case libre
la plus basse. Le premier qui aligne quatre pions (horizontalement,
verticalement ou en diagonale) gagne ; si la grille est pleine sans
alignement, la partie est nulle. Une colonne pleine ne peut plus être
jouée.

### Exercice 1 — Les nombres du jeu

Complétez les constantes. Pour `NB_DIRECTIONS`, comptez les directions
d'alignement différentes (une diagonale dans un sens ou dans l'autre
compte pour une direction chacune ; « gauche » et « droite » sont la
même direction horizontale).

In [ ]:
N_LIGNES = 6
N_COLONNES = 7
NB_DIRECTIONS = 4     # horizontale, verticale, deux diagonales
NB_COUPS_MAX = 42     # 6 x 7 cases, un pion par coup

In [ ]:
assert (N_LIGNES, N_COLONNES) == (6, 7), "la grille fait 6 lignes et 7 colonnes"
assert NB_DIRECTIONS == 4, "horizontale, verticale, et deux diagonales"
assert NB_COUPS_MAX == N_LIGNES * N_COLONNES
print("✅ Exercice 1 réussi")

---

## 2. L'environnement PettingZoo

[PettingZoo](https://pettingzoo.farama.org/environments/classic/connect_four/)
fournit le jeu sous le nom `connect_four_v3`. Les deux joueurs
s'appellent `player_0` (il commence) et `player_1`. Le schéma est
toujours le même : `env.last()` donne ce que voit le joueur dont c'est
le tour, `env.step(action)` joue son coup.

In [ ]:
env = connect_four_v3.env()
env.reset(seed=42)
print("joueurs         :", env.agents)
print("à qui de jouer  :", env.agent_selection)
print("actions         :", env.action_space("player_0"))
print("observations    :", env.observation_space("player_0"))

L'action est un entier de 0 à 6 : la colonne. L'observation est un
dictionnaire à deux clés : `"observation"`, un tableau de forme
`(6, 7, 2)`, et `"action_mask"`, 7 valeurs qui valent 1 pour chaque
colonne encore jouable.

### Exercice 2 — Lire `env.last()`

`env.last()` renvoie cinq valeurs. Récupérez la forme du tableau
d'observation, le masque et le nom du joueur qui doit jouer.

In [ ]:
observation, reward, terminated, truncated, info = env.last()
forme = observation["observation"].shape
masque = observation["action_mask"]
joueur = env.agent_selection
print(forme, masque, joueur)

In [ ]:
assert tuple(forme) == (6, 7, 2)
assert list(masque) == [1] * 7, "au début, les 7 colonnes sont jouables"
assert joueur == "player_0"
print("✅ Exercice 2 réussi")

### L'observation : deux canaux

![Les deux canaux de l'observation](https://raw.githubusercontent.com/racousin/data_science_practice/main/courseware/content/python-medecine/assets/pym-s8-projet-puissance-4/observation-canaux.png)

`observation[:, :, 0]` contient **mes** pions (1 = occupé par moi),
`observation[:, :, 1]` ceux de **l'adversaire**. La ligne 0 est en
**haut**, la ligne 5 en bas. Surtout, la vue est toujours celle du
joueur qui doit jouer : le même code marche que l'on commence ou non.

### Lire du code 1

Que va afficher ce code ? Écrivez votre réponse **avant** d'exécuter.

In [ ]:
env.reset(seed=42)
env.step(3)        # player_0 joue la colonne 3
env.step(3)        # player_1 joue aussi la colonne 3
obs = env.last()[0]["observation"]
print(env.agent_selection)
print(obs[5, 3, 0], obs[4, 3, 0], obs[4, 3, 1])

**Réponse.** `player_0`, puis `1 0 1`. C'est de nouveau à `player_0`,
donc la vue est la sienne : son pion est en bas de la colonne 3
(`[5, 3, 0] == 1`), le pion adverse est juste au-dessus, dans le canal 1
(`[4, 3, 1] == 1`), et rien de « à moi » en ligne 4.

PettingZoo sait aussi dessiner la grille : avec
`render_mode="rgb_array"`, `env.render()` renvoie une image que
matplotlib affiche.

In [ ]:
env_image = connect_four_v3.env(render_mode="rgb_array")
env_image.reset(seed=42)
for col in [3, 3, 4, 2]:
    env_image.step(col)
plt.imshow(env_image.render())
plt.axis("off")
plt.title("Après 4 coups : 3, 3, 4, 2")
plt.show()
env_image.close()

### Exercice 3 — Afficher le plateau en texte

Écrivez `afficher_plateau(observation)` qui renvoie une **chaîne** de
6 lignes, une par ligne de la grille (de haut en bas), les 7 cases
séparées par une espace : `X` pour mes pions, `O` pour ceux de
l'adversaire, `.` pour une case vide. Commencez par
`obs = np.asarray(observation)` : sur la plateforme, l'observation
peut arriver sous forme de listes imbriquées.

In [ ]:
def afficher_plateau(observation):
    """Le plateau en texte : X = mes pions, O = l'adversaire, . = vide."""
    obs = np.asarray(observation)
    lignes = []
    for ligne in range(6):
        cases = []
        for col in range(7):
            if obs[ligne, col, 0] == 1:
                cases.append("X")
            elif obs[ligne, col, 1] == 1:
                cases.append("O")
            else:
                cases.append(".")
        lignes.append(" ".join(cases))
    return "\n".join(lignes)


print(afficher_plateau(obs))

In [ ]:
essai = np.zeros((6, 7, 2), dtype=int)
essai[5, 3, 0] = 1
essai[4, 3, 1] = 1
texte = afficher_plateau(essai)
assert isinstance(texte, str), "renvoyez une chaîne (return), pas un print"
lignes = texte.splitlines()
assert len(lignes) == 6
assert lignes[0] == ". . . . . . ."
assert lignes[4] == ". . . O . . ."
assert lignes[5] == ". . . X . . ."
assert afficher_plateau(essai.tolist()) == texte, "listes imbriquées acceptées"
print("✅ Exercice 3 réussi")

---

## 3. Les coups autorisés

### Trouvez l'erreur

Cette fonction doit renvoyer les **numéros** des colonnes jouables.
Exécutez-la : que renvoie-t-elle ? Pourquoi ?

In [ ]:
def colonnes_libres_bug(action_mask):
    return [autorise for autorise in action_mask if autorise]

print(colonnes_libres_bug([0, 1, 0, 1, 0, 1, 0]))

**Réponse.** `[1, 1, 1]` : la fonction garde les **valeurs** du masque
(des 1), pas leurs **indices**. Un agent qui l'utilise jouerait
toujours la colonne 1, même pleine. `enumerate` donne l'indice et la
valeur.

### Exercice 4 — `colonnes_libres`

Écrivez la version correcte, avec `enumerate`.

In [ ]:
def colonnes_libres(action_mask):
    """Les indices des colonnes autorisées (masque à 1)."""
    return [col for col, autorise in enumerate(action_mask) if autorise]

In [ ]:
assert colonnes_libres([1, 1, 1, 1, 1, 1, 1]) == [0, 1, 2, 3, 4, 5, 6]
assert colonnes_libres([0, 1, 0, 1, 0, 1, 0]) == [1, 3, 5]
assert colonnes_libres(np.array([0, 0, 0, 1, 0, 0, 0], dtype=np.int8)) == [3]
print("✅ Exercice 4 réussi")

### Exercice 5 — Une partie au hasard

Complétez la boucle de jeu de PettingZoo. `env.agent_iter()` donne à
chaque tour le nom du joueur dont c'est le tour. Si sa partie est finie
(`terminated` ou `truncated`), il doit jouer `None` pour sortir ; et
s'il a reçu `reward == 1`, c'est lui le gagnant. Sinon, il tire un coup
autorisé avec `env.action_space(nom).sample(masque)`.

La fonction renvoie `(gagnant, nb_coups)`, `gagnant` valant `None` en
cas de nulle.

In [ ]:
def partie_aleatoire(seed=None):
    """Une partie entre deux joueurs au hasard : (gagnant ou None, nb de coups)."""
    env = connect_four_v3.env()
    env.reset(seed=seed)
    gagnant, nb_coups = None, 0
    for nom in env.agent_iter():
        observation, reward, terminated, truncated, info = env.last()
        if terminated or truncated:
            if reward == 1:
                gagnant = nom
            action = None
        else:
            action = env.action_space(nom).sample(observation["action_mask"])
            nb_coups += 1
        env.step(action)
    env.close()
    return gagnant, nb_coups

In [ ]:
for graine in range(20):
    gagnant, nb_coups = partie_aleatoire(seed=graine)
    assert gagnant in ("player_0", "player_1", None)
    assert 7 <= nb_coups <= 42, "au moins 7 coups pour aligner 4 pions"
    if gagnant is None:
        assert nb_coups == 42, "une nulle remplit la grille"
print("exemple :", partie_aleatoire(seed=1))
print("✅ Exercice 5 réussi")

---

## 4. Le contrat d'agent de ML-Arena

Sur [ML-Arena](https://ml-arena.com/viewchallenge/198), vous ne manipulez pas
l'environnement : la plateforme appelle **votre classe `Agent`**
(fichier `agent.py`). Elle la construit sans argument, puis :

| appel | quand | renvoie |
|---|---|---|
| `setup(observation_space, action_space)` | une fois, avant le match | `True` |
| `reset(env_player_name, episode_index)` | avant **chaque** partie : votre place (`player_0` ou `player_1`) change | `True` |
| `choose_action(observation, reward, terminated, truncated, info, action_mask)` | à chacun de vos tours | une colonne autorisée (`int`) |

`observation` est le tableau `(6, 7, 2)` (déjà extrait du
dictionnaire) et `action_mask` le masque. Si `terminated` ou
`truncated` est vrai, renvoyez `None`. La plateforme coupe un appel qui
dure plus de **0.25 s** : le match est alors perdu.

### Exercice 6 — `AgentAleatoire`

Écrivez la classe : `setup` et `reset` renvoient `True` (`reset`
retient sa place dans `self.env_player_name`), `choose_action` joue
une colonne autorisée au hasard avec `random.choice` et
`colonnes_libres`.

In [ ]:
class AgentAleatoire:
    """Joue une colonne autorisée au hasard (le contrat ML-Arena)."""

    def setup(self, observation_space, action_space):
        return True

    def reset(self, env_player_name, episode_index):
        self.env_player_name = env_player_name
        self.episode_index = episode_index
        return True

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        return random.choice(colonnes_libres(action_mask))

In [ ]:
agent = AgentAleatoire()
assert agent.setup(None, None) is True
assert agent.reset("player_1", 0) is True
assert agent.env_player_name == "player_1"
vide = np.zeros((6, 7, 2), dtype=np.int8)
for _ in range(500):
    masque = np.random.randint(0, 2, size=7)
    masque[np.random.randint(7)] = 1
    action = agent.choose_action(vide, action_mask=masque)
    assert masque[action] == 1, f"colonne {action} interdite par {masque}"
assert agent.choose_action(vide, terminated=True, action_mask=masque) is None
print("✅ Exercice 6 réussi")

---

## 5. Faire jouer deux agents

### Exercice 7 — `jouer_partie`

Voici la boucle qui fait jouer deux agents **comme ML-Arena les
appelle**. Complétez les trois `...` :

1. `places` associe `"player_0"` à l'agent qui commence et
   `"player_1"` à l'autre (selon `a_commence`) ;
2. avant la partie, chaque agent reçoit `reset(nom_de_sa_place, index)` ;
3. à chaque tour, l'agent de la place `nom` choisit son coup :
   `choose_action(observation["observation"], float(reward), False,
   False, info, action_mask=observation["action_mask"])`.

La fonction renvoie le résultat **de `agent_a`** (+1, 0 ou −1) et le
nombre de coups.

In [ ]:
def jouer_partie(agent_a, agent_b, a_commence=True, index=0):
    """Une partie appelée comme sur ML-Arena : (résultat de agent_a, nb de coups).

    Le résultat vaut +1 (agent_a gagne), 0 (nulle) ou -1 (agent_a perd).
    """
    env = connect_four_v3.env()
    env.reset()
    if a_commence:
        places = {"player_0": agent_a, "player_1": agent_b}
    else:
        places = {"player_0": agent_b, "player_1": agent_a}
    for nom, agent in places.items():
        agent.reset(nom, index)
    gains = {nom: 0.0 for nom in env.possible_agents}
    nb_coups = 0
    for nom in env.agent_iter():
        observation, reward, terminated, truncated, info = env.last()
        if terminated or truncated:
            env.step(None)  # un joueur dont la partie est finie sort ainsi
            continue
        action = places[nom].choose_action(
            observation["observation"], float(reward), False, False, info,
            action_mask=observation["action_mask"])
        env.step(action)
        nb_coups += 1
        for joueur, gain in env.rewards.items():
            gains[joueur] += gain
    env.close()
    nom_a = "player_0" if a_commence else "player_1"
    return gains[nom_a], nb_coups

In [ ]:
class _Espion(AgentAleatoire):
    def reset(self, env_player_name, episode_index):
        self.places = getattr(self, "places", []) + [env_player_name]
        return super().reset(env_player_name, episode_index)

espion = _Espion()
for i in range(10):
    resultat, nb_coups = jouer_partie(espion, AgentAleatoire(),
                                      a_commence=(i % 2 == 0), index=i)
    assert resultat in (-1, 0, 1)
    assert 7 <= nb_coups <= 42
assert espion.places == ["player_0", "player_1"] * 5, "reset avant chaque partie"
print("✅ Exercice 7 réussi")

### Lire du code 2

`Espion` hérite d'`AgentAleatoire` et affiche chaque appel à `reset`.
Que va afficher ce code ?

In [ ]:
class Espion(AgentAleatoire):
    def reset(self, env_player_name, episode_index):
        print("reset", env_player_name, episode_index)
        return super().reset(env_player_name, episode_index)

espion = Espion()
for i in range(3):
    jouer_partie(espion, AgentAleatoire(), a_commence=(i % 2 == 0), index=i)

**Réponse.** `reset player_0 0`, `reset player_1 1`, `reset player_0 2`.
L'espion commence une partie sur deux. Sur ML-Arena aussi la place
change d'une partie à l'autre : un agent qui retiendrait sa place une
fois pour toutes dans `setup` jouerait avec la mauvaise.

### Exercice 8 — `jouer_match`

Écrivez `jouer_match(agent_a, agent_b, n_parties)` : appelle `setup`
une fois sur chaque agent (avec les espaces de `player_0`), puis joue
`n_parties` parties, `agent_a` commençant les parties d'indice pair.
Elle renvoie un dictionnaire `{"victoires", "nuls", "defaites",
"coups"}` vu par `agent_a`, `"coups"` étant la liste des longueurs.

In [ ]:
def jouer_match(agent_a, agent_b, n_parties=100):
    """n parties, agent_a commence une partie sur deux. Bilan vu par agent_a."""
    env = connect_four_v3.env()
    for agent in (agent_a, agent_b):  # une fois par match, comme sur ML-Arena
        agent.setup(env.observation_space("player_0"),
                    env.action_space("player_0"))
    bilan = {"victoires": 0, "nuls": 0, "defaites": 0, "coups": []}
    for i in range(n_parties):
        resultat, nb_coups = jouer_partie(agent_a, agent_b,
                                          a_commence=(i % 2 == 0), index=i)
        if resultat > 0:
            bilan["victoires"] += 1
        elif resultat < 0:
            bilan["defaites"] += 1
        else:
            bilan["nuls"] += 1
        bilan["coups"].append(nb_coups)
    return bilan

In [ ]:
bilan = jouer_match(AgentAleatoire(), AgentAleatoire(), 30)
assert bilan["victoires"] + bilan["nuls"] + bilan["defaites"] == 30
assert len(bilan["coups"]) == 30
print({k: v for k, v in bilan.items() if k != "coups"})
print("✅ Exercice 8 réussi")

### Ce que donne le hasard

Deux joueurs au hasard sur 200 parties, puis 300 parties où le même
joueur commence toujours. Avant d'exécuter : le joueur qui commence
a-t-il un avantage ? Combien de coups dure une partie ? Les nulles
sont-elles fréquentes ?

In [ ]:
bilan = jouer_match(AgentAleatoire(), AgentAleatoire(), 200)
coups = np.array(bilan["coups"])
print("victoires / nuls / défaites :", bilan["victoires"], bilan["nuls"],
      bilan["defaites"])
print(f"durée : moyenne {coups.mean():.1f} coups, min {coups.min()}, max {coups.max()}")

premier = [jouer_partie(AgentAleatoire(), AgentAleatoire(), a_commence=True)[0]
           for _ in range(300)]
print(f"le joueur qui commence gagne {np.mean(np.array(premier) > 0):.0%} des parties")

plt.hist(coups, bins=range(7, 44), color="#2f5d9e")
plt.xlabel("nombre de coups")
plt.ylabel("parties")
plt.title("Durée de 200 parties entre deux agents aléatoires")
plt.show()

**Lecture.** Entre deux agents aléatoires, celui qui commence gagne
nettement plus d'une partie sur deux (environ 55 %) : il a un pion
d'avance. C'est pourquoi les places alternent dans un match. Une
partie dure une vingtaine de coups ; les nulles sont rares (il faut
remplir les 42 cases). Même deux agents identiques ne font pas
exactement 50/50 sur 200 parties : c'est la fluctuation
d'échantillonnage, que le lab 3 chiffre.

---

## 6. Un hasard plus malin

### Exercice 9 — `AgentPondere`, par héritage

Les colonnes du centre participent à plus d'alignements. Écrivez
`AgentPondere`, qui **hérite** d'`AgentAleatoire` et ne redéfinit que
`choose_action` : il tire une colonne autorisée avec
`random.choices(colonnes, weights=poids)`, les poids étant
`[1, 2, 3, 4, 3, 2, 1]` (colonne 0 à 6).

In [ ]:
class AgentPondere(AgentAleatoire):
    """Au hasard, mais les colonnes du centre sont plus probables."""

    POIDS = [1, 2, 3, 4, 3, 2, 1]

    def choose_action(self, observation, reward=0.0, terminated=False,
                      truncated=False, info=None, action_mask=None):
        if terminated or truncated:
            return None
        colonnes = colonnes_libres(action_mask)
        poids = [self.POIDS[col] for col in colonnes]
        return random.choices(colonnes, weights=poids)[0]

In [ ]:
agent = AgentPondere()
assert isinstance(agent, AgentAleatoire), "AgentPondere hérite d'AgentAleatoire"
assert agent.reset("player_0", 0) is True, "reset est hérité"
vide = np.zeros((6, 7, 2), dtype=np.int8)
tirages = [agent.choose_action(vide, action_mask=[1] * 7) for _ in range(3000)]
assert tirages.count(3) > 2 * tirages.count(0), "le centre doit sortir plus souvent"
assert all(agent.choose_action(vide, action_mask=[1, 0, 0, 0, 0, 0, 1]) in (0, 6)
           for _ in range(100))
bilan = jouer_match(AgentPondere(), AgentAleatoire(), 100)
print("pondéré contre aléatoire :", bilan["victoires"], "victoires sur 100")
print("✅ Exercice 9 réussi")

---

## 7. Pour aller plus loin (facultatif)

### Exercice 10 — Le film d'une partie

Écrivez `images_partie(agent_a, agent_b)` : comme `jouer_partie`, mais
avec `render_mode="rgb_array"`, et renvoie la liste des images
(`env.render()`) prises après chaque coup. Affichez-en six.

In [ ]:
def images_partie(agent_a, agent_b):
    """Les images de la grille après chaque coup d'une partie."""
    env = connect_four_v3.env(render_mode="rgb_array")
    env.reset()
    places = {"player_0": agent_a, "player_1": agent_b}
    for nom, agent in places.items():
        agent.reset(nom, 0)
    images = []
    for nom in env.agent_iter():
        observation, reward, terminated, truncated, info = env.last()
        if terminated or truncated:
            env.step(None)
            continue
        env.step(places[nom].choose_action(
            observation["observation"], float(reward), False, False, info,
            action_mask=observation["action_mask"]))
        images.append(env.render())
    env.close()
    return images

In [ ]:
images = images_partie(AgentPondere(), AgentAleatoire())
assert len(images) >= 7 and images[0].ndim == 3
choix = np.linspace(0, len(images) - 1, 6).astype(int)
fig, axes = plt.subplots(1, 6, figsize=(15, 2.6))
for ax, i in zip(axes, choix):
    ax.imshow(images[i])
    ax.set_title(f"coup {i + 1}")
    ax.axis("off")
plt.show()
print("✅ Exercice 10 réussi")

---

**Bilan.** Vous savez lire l'observation, jouer un coup autorisé et
faire s'affronter deux agents comme le fait ML-Arena. Le lab 2
construit un agent qui réfléchit : gagner quand c'est possible, bloquer
l'adversaire, ne pas lui offrir la victoire.